# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# DINOv2 extension: three-model attribution study

**Core comparison:** ResNet + Grad-CAM versus DINOv2 + gradient-weighted attention rollout versus UNI + gradient-weighted attention rollout.

This notebook adds the missing DINOv2 ViT-L/14 arm while reusing the grouped OOF folds and frozen faithfulness cohort from notebook 04. UNI and DINOv2 use the same Transformer explanation algorithm. Highlighted image regions are contributors to model predictions, not evidence of biological causality.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f'GPU {index}:', torch.cuda.get_device_name(index))

In [ ]:
def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
DATASET_DIR = (
    PROJECT_ROOT / 'Colorectal Histology MNIST'
    / 'Kather_texture_2016_image_tiles_5000'
    / 'Kather_texture_2016_image_tiles_5000'
)
BASE_ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'grouped_oof_faithfulness'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'dinov2_three_model'
DINO_CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints' / 'dinov2'
RESNET_CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints' / 'resnet18'
FEATURE_DIR = ARTIFACT_DIR / 'features'
TOKEN_DIR = ARTIFACT_DIR / 'patch_tokens'
PROOF_DIR = ARTIFACT_DIR / 'proof_fold'
CLASSIFICATION_DIR = ARTIFACT_DIR / 'classification'
FAITHFULNESS_DIR = ARTIFACT_DIR / 'faithfulness'
STABILITY_DIR = ARTIFACT_DIR / 'stability'
FIGURE_DIR = ARTIFACT_DIR / 'figures'
for directory in (
    DINO_CHECKPOINT_DIR, RESNET_CHECKPOINT_DIR, FEATURE_DIR, TOKEN_DIR, PROOF_DIR,
    CLASSIFICATION_DIR, FAITHFULNESS_DIR, STABILITY_DIR, FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)
print('Artifacts:', ARTIFACT_DIR)

In [ ]:
from Methods.BaselineCNN import discover_images, set_seed
from Methods.CNNBenchmark import build_resnet18
from Methods.DINOv2Attribution import (
    build_dinov2_classifier,
    cache_dinov2_patch_tokens,
    resolve_dinov2_transform,
)
from Methods.GroupAwareEvaluation import (
    attach_model_predictions,
    classification_metrics,
    create_main_figures,
    create_three_model_heatmaps,
    evaluate_cnn_faithfulness,
    evaluate_cnn_stability,
    evaluate_dinov2_faithfulness,
    evaluate_dinov2_stability,
    extract_all_transformer_features,
    pairwise_model_comparisons,
    run_cnn_oof,
    run_transformer_oof,
    save_classification_artifacts,
    validate_oof_assignments,
    within_model_random_deletion_tests,
)
from Methods.UNIAttribution import UNIImageDataset

REFERENCE_SEED = 41
ALL_SEEDS = (11, 23, 41, 57, 73, 89, 101, 131, 151, 181)
PROOF_FOLD = 0
DEVICE = torch.device('cuda:1' if torch.cuda.device_count() > 1 else ('cuda:0' if torch.cuda.is_available() else 'cpu'))
NUM_WORKERS = 4
IMAGE_BATCH_SIZE = 16
FEATURE_BATCH_SIZE = 128
EPOCHS = 40
LEARNING_RATE = 1e-3
PATIENCE = 8

# Complete the proof stage first. The full-stage flags should remain False
# until the proof assertions and saved output tables have been inspected.
RUN_PROOF_PATCH_TOKENS = True
RUN_PROOF_ATTRIBUTION = True
RUN_FULL_ONE_SEED_OOF = False
RUN_FULL_FAITHFULNESS = True
RUN_COHORT_PATCH_TOKENS = True
RUN_MULTI_SEED_OOF = True
RUN_STABILITY = True
RUN_RESNET_OOF = True
FORCE_FEATURE_REBUILD = False
FORCE_MODEL_RETRAIN = False
FORCE_FAITHFULNESS_REBUILD = False
set_seed(REFERENCE_SEED)
print('Device:', DEVICE)

## 1. Reuse the exact grouped folds and frozen cohort

The cohort image selection is never rerun. DINOv2 predictions are attached to the existing manifest by image path after OOF inference.

In [ ]:
manifest = discover_images(DATASET_DIR)
class_table = manifest[['class_name', 'label']].drop_duplicates().sort_values('label')
CLASS_NAMES = class_table['class_name'].tolist()
assignment_path = BASE_ARTIFACT_DIR / 'folds' / 'fold_assignments.csv'
cohort_path = BASE_ARTIFACT_DIR / 'faithfulness_cohort_manifest.csv'
existing_prediction_path = (
    BASE_ARTIFACT_DIR / 'classification' / 'oof_predictions_and_logits.csv'
)
for required_path in (assignment_path, cohort_path, existing_prediction_path):
    if not required_path.is_file():
        raise FileNotFoundError(f'Run notebook 04 first; missing {required_path}')
assignments = pd.read_csv(assignment_path)
frozen_cohort = pd.read_csv(cohort_path)
existing_predictions = pd.read_csv(existing_prediction_path)
validate_oof_assignments(assignments, expected_image_count=len(manifest))
assert frozen_cohort['label'].nunique() == 8
assert set(existing_predictions['model']) >= {'UNI', 'CNN'}
print('Images:', len(manifest))
print('Frozen cohort:', len(frozen_cohort))
display(pd.crosstab(frozen_cohort['class_name'], frozen_cohort['cohort_stratum']))

## 2. Attribution algorithm audit

The current Transformer method is **class-conditioned gradient-weighted attention rollout**, not full Chefer relevance propagation. Both UNI and DINOv2 use the same implementation: target-logit attention gradients, positive attention-gradient products, head averaging, residual identity, and layerwise rollout. Raw attention and ordinary rollout remain weak baselines.

In [ ]:
audit_path = PROJECT_ROOT / 'Methods' / 'DINOv2Attribution' / 'ATTRIBUTION_AUDIT.md'
print(audit_path.read_text())

## 3. Load frozen DINOv2 ViT-L/14 and cache CLS features

The encoder is instantiated at 224×224, yielding 256 patch tokens on a native 16×16 grid. The classifier is the same LayerNorm–dropout–linear head used by UNI.

In [ ]:
dinov2_model = build_dinov2_classifier(
    num_classes=len(CLASS_NAMES),
    device=DEVICE,
)
dinov2_transform, dinov2_data_config = resolve_dinov2_transform(
    dinov2_model.encoder
)
DINO_PREPROCESSING_ID = (
    'vit_large_patch14_dinov2.lvd142m|224|'
    + repr(sorted(dinov2_data_config.items()))
)
dinov2_feature_cache = extract_all_transformer_features(
    dinov2_model,
    manifest,
    DEVICE,
    FEATURE_DIR / 'dinov2_cls_all_images.pt',
    image_transform=dinov2_transform,
    preprocessing_id=DINO_PREPROCESSING_ID,
    batch_size=IMAGE_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    overwrite=FORCE_FEATURE_REBUILD,
)
sanity_dataset = UNIImageDataset(
    assignments.query("fold == @PROOF_FOLD and split == 'test'").head(1),
    image_transform=dinov2_transform,
)
sanity_image = sanity_dataset[0]['image'].unsqueeze(0).to(DEVICE)
with torch.inference_mode():
    sanity_tokens = dinov2_model.forward_tokens(sanity_image)
assert sanity_tokens.shape[1] == 1 + 16 * 16, sanity_tokens.shape
assert dinov2_model.feature_dim == 1024
print('Feature cache:', tuple(dinov2_feature_cache['features'].shape))
print('Token tensor:', tuple(sanity_tokens.shape))
print('Data config:', dinov2_data_config)
run_metadata = {
    'dinov2_model': 'vit_large_patch14_dinov2.lvd142m',
    'image_size': 224,
    'patch_size': 14,
    'native_grid_size': 16,
    'evaluation_grid_size': 14,
    'feature_dim': dinov2_model.feature_dim,
    'reference_seed': REFERENCE_SEED,
    'all_seeds': list(ALL_SEEDS),
    'proof_fold': PROOF_FOLD,
    'transformer_attribution': 'class-conditioned gradient-weighted attention rollout',
    'dinov2_data_config': {key: str(value) for key, value in dinov2_data_config.items()},
}
(ARTIFACT_DIR / 'run_metadata.json').write_text(
    json.dumps(run_metadata, indent=2), encoding='utf-8'
)

## 4. Proof of concept: one fold and seed

This stage trains fold 0, seed 41 only. Its prediction and attribution files must match the required UNI schema before full OOF training is enabled.

In [ ]:
proof_assignments = assignments[assignments['fold'] == PROOF_FOLD].copy()
proof_predictions, proof_history = run_transformer_oof(
    dinov2_model,
    dinov2_feature_cache,
    manifest,
    proof_assignments,
    CLASS_NAMES,
    DEVICE,
    DINO_CHECKPOINT_DIR,
    seeds=(REFERENCE_SEED,),
    batch_size=FEATURE_BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    force_retrain=FORCE_MODEL_RETRAIN,
    model_name='DINOv2',
)
proof_predictions.to_csv(PROOF_DIR / 'dinov2_predictions_and_logits.csv', index=False)
proof_history.to_csv(PROOF_DIR / 'dinov2_training_history.csv', index=False)
proof_metrics, proof_per_class, proof_matrices = classification_metrics(
    proof_predictions, CLASS_NAMES
)
proof_metrics.to_csv(PROOF_DIR / 'dinov2_classification_metrics.csv', index=False)
proof_per_class.to_csv(PROOF_DIR / 'dinov2_per_class_accuracy.csv', index=False)
assert len(proof_predictions) == len(proof_assignments.query("split == 'test'"))
assert proof_predictions['model'].eq('DINOv2').all()
assert not proof_predictions.duplicated('relative_path').any()
assert any(column.startswith('logit_') for column in proof_predictions)
assert any(column.startswith('probability_') for column in proof_predictions)
display(proof_metrics.style.format(precision=4))

In [ ]:
if RUN_PROOF_PATCH_TOKENS:
    proof_test_frame = proof_assignments.query("split == 'test'").copy()
    proof_token_dataset = UNIImageDataset(
        proof_test_frame,
        augment=False,
        image_transform=dinov2_transform,
    )
    proof_token_loader = DataLoader(
        proof_token_dataset,
        batch_size=IMAGE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
    )
    token_path, token_index_path = cache_dinov2_patch_tokens(
        dinov2_model,
        proof_token_loader,
        TOKEN_DIR / f'dinov2_fold_{PROOF_FOLD}_patch_tokens_float16.npy',
        TOKEN_DIR / f'dinov2_fold_{PROOF_FOLD}_patch_token_index.csv',
        DEVICE,
        overwrite=False,
    )
    print('Patch tokens:', token_path)
    print('Token index:', token_index_path)

In [ ]:
proof_cohort = frozen_cohort[frozen_cohort['fold'] == PROOF_FOLD].copy()
proof_cohort = attach_model_predictions(
    proof_cohort,
    proof_predictions,
    model_name='DINOv2',
    reference_seed=REFERENCE_SEED,
    column_prefix='dinov2',
)
proof_cohort.to_csv(PROOF_DIR / 'proof_cohort_with_dinov2.csv', index=False)
if RUN_PROOF_ATTRIBUTION:
    proof_faithfulness, proof_curves = evaluate_dinov2_faithfulness(
        dinov2_model,
        proof_cohort,
        DINO_CHECKPOINT_DIR,
        REFERENCE_SEED,
        DEVICE,
        PROOF_DIR,
        image_transform=dinov2_transform,
        random_repeats=20,
        heatmap_limit=8,
    )
else:
    proof_faithfulness = pd.read_csv(PROOF_DIR / 'dinov2_faithfulness_metrics.csv')
    proof_curves = pd.read_csv(PROOF_DIR / 'dinov2_deletion_curves.csv')

required_metric_columns = {
    'attribution_occlusion_spearman', 'top_target_logit_auc',
    'random_target_logit_auc', 'bottom_target_logit_auc',
    'top_minus_random_target_logit_auc',
    'top_minus_random_margin_auc', 'top_5_target_logit_drop',
    'top_10_target_logit_drop', 'top_20_target_logit_drop',
    'top_30_target_logit_drop', 'top_5_margin_drop',
    'top_10_margin_drop', 'top_20_margin_drop', 'top_30_margin_drop',
}
required_curve_columns = {
    'predicted_class_logit', 'true_class_logit',
    'target_class_logit', 'target_margin', 'strategy',
    'fraction_removed',
}
assert required_metric_columns <= set(proof_faithfulness.columns)
assert required_curve_columns <= set(proof_curves.columns)
assert set(proof_faithfulness['method']) == {
    'raw_attention', 'attention_rollout', 'gradient_attention_rollout'
}
assert set(proof_curves['strategy']) == {'top', 'random', 'bottom'}
proof_maps = pd.read_csv(PROOF_DIR / 'dinov2_attribution_maps.csv')
assert proof_maps['native_grid_size'].eq(16).all()
assert proof_maps['evaluation_grid_size'].eq(14).all()
print('Proof-of-concept schema and shared-grid checks passed.')

## 5. Full DINOv2 OOF and frozen-cohort evaluation

Enable the full one-seed stage only after the proof outputs pass. Multi-seed training remains a separate final stage.

In [ ]:
full_seeds = ALL_SEEDS if RUN_MULTI_SEED_OOF else (REFERENCE_SEED,)
if RUN_FULL_ONE_SEED_OOF or RUN_MULTI_SEED_OOF:
    dinov2_predictions, dinov2_history = run_transformer_oof(
        dinov2_model,
        dinov2_feature_cache,
        manifest,
        assignments,
        CLASS_NAMES,
        DEVICE,
        DINO_CHECKPOINT_DIR,
        seeds=full_seeds,
        batch_size=FEATURE_BATCH_SIZE,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        patience=PATIENCE,
        force_retrain=FORCE_MODEL_RETRAIN,
        model_name='DINOv2',
    )
    dinov2_predictions.to_csv(
        CLASSIFICATION_DIR / 'dinov2_oof_predictions_and_logits.csv', index=False
    )
    dinov2_history.to_csv(
        CLASSIFICATION_DIR / 'dinov2_training_history.csv', index=False
    )
else:
    full_prediction_path = CLASSIFICATION_DIR / 'dinov2_oof_predictions_and_logits.csv'
    dinov2_predictions = (
        pd.read_csv(full_prediction_path) if full_prediction_path.is_file()
        else pd.DataFrame()
    )

if RUN_RESNET_OOF or RUN_FULL_ONE_SEED_OOF or RUN_MULTI_SEED_OOF:
    resnet_predictions, resnet_history = run_cnn_oof(
        assignments,
        CLASS_NAMES,
        DEVICE,
        RESNET_CHECKPOINT_DIR,
        seeds=full_seeds,
        batch_size=32,
        num_workers=NUM_WORKERS,
        epochs=20,
        learning_rate=1e-4,
        patience=PATIENCE,
        force_retrain=FORCE_MODEL_RETRAIN,
        model_builder=lambda: build_resnet18(
            num_classes=len(CLASS_NAMES), pretrained=True, freeze_backbone=False
        ),
        model_name='ResNet18',
    )
    resnet_predictions.to_csv(
        CLASSIFICATION_DIR / 'resnet18_oof_predictions_and_logits.csv', index=False
    )
    resnet_history.to_csv(
        CLASSIFICATION_DIR / 'resnet18_training_history.csv', index=False
    )
else:
    resnet_prediction_path = CLASSIFICATION_DIR / 'resnet18_oof_predictions_and_logits.csv'
    resnet_predictions = (
        pd.read_csv(resnet_prediction_path) if resnet_prediction_path.is_file()
        else pd.DataFrame()
    )

if not dinov2_predictions.empty and not resnet_predictions.empty:
    assert dinov2_predictions.groupby('seed').size().eq(len(manifest)).all()
    assert dinov2_predictions.groupby('seed')['class_name'].nunique().eq(8).all()
    assert resnet_predictions.groupby('seed').size().eq(len(manifest)).all()
    assert resnet_predictions.groupby('seed')['class_name'].nunique().eq(8).all()
    retained_uni_predictions = existing_predictions[
        (existing_predictions['model'] == 'UNI')
        & (existing_predictions['seed'].isin(full_seeds))
    ]
    missing_uni_seeds = set(full_seeds) - set(retained_uni_predictions['seed'])
    if missing_uni_seeds:
        raise RuntimeError(
            f'Notebook 04 is missing UNI OOF seeds: {sorted(missing_uni_seeds)}'
        )
    all_predictions = pd.concat(
        (retained_uni_predictions, resnet_predictions, dinov2_predictions),
        ignore_index=True,
    )
    classification_results, per_class_results = save_classification_artifacts(
        all_predictions, CLASS_NAMES, CLASSIFICATION_DIR
    )
    display(classification_results.query("scope == 'aggregate_oof'").style.format(precision=4))

In [ ]:
def faithfulness_artifact_paths(output_dir, prefix):
    return {
        'metrics': output_dir / f'{prefix}_faithfulness_metrics.csv',
        'curves': output_dir / f'{prefix}_deletion_curves.csv',
        'maps': output_dir / f'{prefix}_attribution_maps.csv',
        'occlusion': output_dir / f'{prefix}_patch_occlusion_scores.csv',
    }


dinov2_faithfulness_paths = faithfulness_artifact_paths(FAITHFULNESS_DIR, 'dinov2')
resnet_faithfulness_paths = faithfulness_artifact_paths(FAITHFULNESS_DIR, 'resnet18')
uni_faithfulness_paths = faithfulness_artifact_paths(
    BASE_ARTIFACT_DIR / 'faithfulness', 'uni'
)
force_faithfulness_rebuild = bool(
    globals().get('FORCE_FAITHFULNESS_REBUILD', False)
)

if RUN_FULL_FAITHFULNESS:
    if dinov2_predictions.empty:
        raise RuntimeError('Complete full one-seed DINOv2 OOF inference first')
    if resnet_predictions.empty:
        raise RuntimeError('Complete full one-seed ResNet18 OOF inference first')
    cohort = attach_model_predictions(
        frozen_cohort,
        dinov2_predictions,
        model_name='DINOv2',
        reference_seed=REFERENCE_SEED,
        column_prefix='dinov2',
    )
    cohort = attach_model_predictions(
        cohort,
        resnet_predictions,
        model_name='ResNet18',
        reference_seed=REFERENCE_SEED,
        column_prefix='resnet18',
    )
    cohort.to_csv(ARTIFACT_DIR / 'three_model_faithfulness_cohort.csv', index=False)
    if RUN_COHORT_PATCH_TOKENS:
        cohort_token_frame = cohort.copy()
        if 'image_path' not in cohort_token_frame.columns:
            cohort_token_frame['image_path'] = cohort_token_frame['path']
        cohort_token_dataset = UNIImageDataset(
            cohort_token_frame, augment=False, image_transform=dinov2_transform
        )
        cohort_token_loader = DataLoader(
            cohort_token_dataset,
            batch_size=IMAGE_BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
            persistent_workers=NUM_WORKERS > 0,
        )
        cache_dinov2_patch_tokens(
            dinov2_model,
            cohort_token_loader,
            TOKEN_DIR / 'dinov2_faithfulness_cohort_patch_tokens_float16.npy',
            TOKEN_DIR / 'dinov2_faithfulness_cohort_patch_token_index.csv',
            DEVICE,
            overwrite=False,
        )
    dinov2_complete = all(path.is_file() for path in dinov2_faithfulness_paths.values())
    if force_faithfulness_rebuild or not dinov2_complete:
        dinov2_faithfulness, dinov2_curves = evaluate_dinov2_faithfulness(
            dinov2_model, cohort, DINO_CHECKPOINT_DIR, REFERENCE_SEED,
            DEVICE, FAITHFULNESS_DIR, image_transform=dinov2_transform,
            random_repeats=20, heatmap_limit=32,
        )
    else:
        print('Reusing complete DINOv2 faithfulness artifacts')
        dinov2_faithfulness = pd.read_csv(dinov2_faithfulness_paths['metrics'])
        dinov2_curves = pd.read_csv(dinov2_faithfulness_paths['curves'])

    resnet_complete = all(path.is_file() for path in resnet_faithfulness_paths.values())
    if force_faithfulness_rebuild or not resnet_complete:
        resnet_faithfulness, resnet_curves = evaluate_cnn_faithfulness(
            cohort,
            RESNET_CHECKPOINT_DIR,
            REFERENCE_SEED,
            DEVICE,
            FAITHFULNESS_DIR,
            num_classes=len(CLASS_NAMES),
            random_repeats=20,
            heatmap_limit=32,
            model_builder=lambda: build_resnet18(
                num_classes=len(CLASS_NAMES), pretrained=False, freeze_backbone=False
            ),
            target_layer_getter=lambda model: model.layer4[-1],
            model_name='ResNet18',
            column_prefix='resnet18',
            output_prefix='resnet18',
        )
    else:
        print('Reusing complete ResNet18 faithfulness artifacts')
        resnet_faithfulness = pd.read_csv(resnet_faithfulness_paths['metrics'])
        resnet_curves = pd.read_csv(resnet_faithfulness_paths['curves'])
else:
    extended_cohort_path = ARTIFACT_DIR / 'three_model_faithfulness_cohort.csv'
    cohort = pd.read_csv(extended_cohort_path) if extended_cohort_path.is_file() else pd.DataFrame()

required_faithfulness_paths = {
    'ResNet18 metrics': resnet_faithfulness_paths['metrics'],
    'ResNet18 curves': resnet_faithfulness_paths['curves'],
    'UNI metrics': uni_faithfulness_paths['metrics'],
    'UNI curves': uni_faithfulness_paths['curves'],
    'DINOv2 metrics': dinov2_faithfulness_paths['metrics'],
    'DINOv2 curves': dinov2_faithfulness_paths['curves'],
}
missing_faithfulness_paths = {
    name: path for name, path in required_faithfulness_paths.items()
    if not path.is_file()
}

if not cohort.empty and not missing_faithfulness_paths:
    metric_paths = {
        'ResNet18': resnet_faithfulness_paths['metrics'],
        'UNI': uni_faithfulness_paths['metrics'],
        'DINOv2': dinov2_faithfulness_paths['metrics'],
    }
    curve_paths = {
        'ResNet18': resnet_faithfulness_paths['curves'],
        'UNI': uni_faithfulness_paths['curves'],
        'DINOv2': dinov2_faithfulness_paths['curves'],
    }
    faithfulness_metrics = pd.concat(
        [pd.read_csv(path) for path in metric_paths.values()], ignore_index=True
    )
    deletion_curves = pd.concat(
        [pd.read_csv(path) for path in curve_paths.values()], ignore_index=True
    )
    faithfulness_metrics.to_csv(FAITHFULNESS_DIR / 'three_model_faithfulness_metrics.csv', index=False)
    deletion_curves.to_csv(FAITHFULNESS_DIR / 'three_model_deletion_curves.csv', index=False)
    primary_methods = {
        'ResNet18': 'gradcam',
        'DINOv2': 'gradient_attention_rollout',
        'UNI': 'gradient_attention_rollout',
    }
    primary_faithfulness = faithfulness_metrics[
        faithfulness_metrics['method']
        == faithfulness_metrics['model'].map(primary_methods)
    ]
    aggregate_faithfulness = (
        primary_faithfulness.groupby(
            ['model', 'class_name', 'correct', 'target_role'], dropna=False
        )
        .agg(
            images=('cohort_id', 'nunique'),
            mean_occlusion_spearman=('attribution_occlusion_spearman', 'mean'),
            mean_top_minus_random_logit_auc=(
                'top_minus_random_target_logit_auc', 'mean'
            ),
            mean_top_minus_random_margin_auc=(
                'top_minus_random_margin_auc', 'mean'
            ),
        )
        .reset_index()
    )
    aggregate_faithfulness.to_csv(
        FAITHFULNESS_DIR / 'three_model_aggregate_faithfulness.csv', index=False
    )
elif not cohort.empty:
    faithfulness_metrics = pd.DataFrame()
    deletion_curves = pd.DataFrame()
    print('Three-model aggregation skipped because these artifacts are missing:')
    for name, path in missing_faithfulness_paths.items():
        print(f'  {name}: {path}')
    print('Set RUN_FULL_FAITHFULNESS=True and rerun this cell to resume missing work.')

## 6. Prespecified co-primary outcomes and statistical tests

Co-primary outcomes are patchwise attribution–occlusion Spearman correlation and within-model top-minus-random true-class-margin AUC. Raw logit outcomes are secondary because model logit scales differ.

In [ ]:
if not cohort.empty and not faithfulness_metrics.empty:
    within_model_tests = within_model_random_deletion_tests(
        faithfulness_metrics,
        bootstrap_iterations=5000,
        confidence=0.95,
        random_seed=2027,
    )
    pairwise_tests, paired_values = pairwise_model_comparisons(
        faithfulness_metrics,
        bootstrap_iterations=5000,
        confidence=0.95,
        random_seed=2027,
    )
    within_model_tests.to_csv(
        FAITHFULNESS_DIR / 'within_model_tests_against_random.csv', index=False
    )
    pairwise_tests.to_csv(
        FAITHFULNESS_DIR / 'three_model_paired_comparisons.csv', index=False
    )
    paired_values.to_csv(
        FAITHFULNESS_DIR / 'three_model_paired_values.csv', index=False
    )
    display(within_model_tests.query("subgroup == 'all'").style.format(precision=4))
    display(
        pairwise_tests.query(
            "comparison_target == 'true_class' and subgroup == 'all' and "
            "metric in ['attribution_occlusion_spearman', "
            "'top_minus_random_margin_auc']"
        ).style.format(precision=4)
    )

## 7. Multi-seed DINOv2 stability

Enable only after every DINO fold checkpoint exists for every predefined seed. Predicted-class maps are compared only when seed pairs predict the same class; common true-class maps are used otherwise.

In [ ]:
if RUN_STABILITY:
    missing = [
        (seed, fold)
        for seed in ALL_SEEDS
        for fold in sorted(assignments['fold'].unique())
        if not (DINO_CHECKPOINT_DIR / f'seed_{seed}' / f'fold_{fold}.pt').is_file()
    ]
    missing_resnet = [
        (seed, fold)
        for seed in ALL_SEEDS
        for fold in sorted(assignments['fold'].unique())
        if not (RESNET_CHECKPOINT_DIR / f'seed_{seed}' / f'fold_{fold}.pt').is_file()
    ]
    missing_uni = [
        (seed, fold)
        for seed in ALL_SEEDS
        for fold in sorted(assignments['fold'].unique())
        if not (
            BASE_ARTIFACT_DIR / 'checkpoints' / 'uni'
            / f'seed_{seed}' / f'fold_{fold}.pt'
        ).is_file()
    ]
    if missing or missing_resnet or missing_uni:
        raise RuntimeError(
            f'Missing checkpoints: DINOv2={len(missing)}, '
            f'ResNet18={len(missing_resnet)}, UNI={len(missing_uni)}'
        )
    _, _, dinov2_stability = evaluate_dinov2_stability(
        dinov2_model,
        cohort,
        DINO_CHECKPOINT_DIR,
        ALL_SEEDS,
        DEVICE,
        STABILITY_DIR,
        image_transform=dinov2_transform,
    )
    _, _, resnet_stability = evaluate_cnn_stability(
        cohort,
        RESNET_CHECKPOINT_DIR,
        ALL_SEEDS,
        DEVICE,
        STABILITY_DIR,
        num_classes=len(CLASS_NAMES),
        model_builder=lambda: build_resnet18(
            num_classes=len(CLASS_NAMES), pretrained=False, freeze_backbone=False
        ),
        target_layer_getter=lambda model: model.layer4[-1],
        model_name='ResNet18',
        output_prefix='resnet18',
    )
else:
    stability_path = STABILITY_DIR / 'dinov2_stability_per_image.csv'
    dinov2_stability = pd.read_csv(stability_path) if stability_path.is_file() else pd.DataFrame()
    resnet_stability_path = STABILITY_DIR / 'resnet18_stability_per_image.csv'
    resnet_stability = (
        pd.read_csv(resnet_stability_path)
        if resnet_stability_path.is_file() else pd.DataFrame()
    )

## 8. Final three-model figures

ResNet-versus-Transformer comparisons evaluate complete model–explanation pipelines because both architecture and explanation algorithm change.

In [ ]:
if not cohort.empty and not faithfulness_metrics.empty:
    existing_stability_path = BASE_ARTIFACT_DIR / 'stability' / 'uni_stability_per_image.csv'
    existing_stability = (
        pd.read_csv(existing_stability_path)
        if existing_stability_path.is_file() else pd.DataFrame()
    )
    stability_metrics = pd.concat(
        [frame for frame in (resnet_stability, dinov2_stability, existing_stability) if not frame.empty],
        ignore_index=True,
    ) if (not existing_stability.empty or not dinov2_stability.empty) else pd.DataFrame()
    if not stability_metrics.empty:
        stability_summary = (
            stability_metrics.groupby(['model', 'class_name', 'cohort_stratum'])
            .agg(
                images=('cohort_id', 'nunique'),
                prediction_agreement=('pairwise_prediction_agreement', 'mean'),
                conditional_map_spearman=(
                    'mean_predicted_class_spearman_same_prediction', 'mean'
                ),
                common_target_spearman=('mean_common_true_class_spearman', 'mean'),
            )
            .reset_index()
        )
        stability_summary.to_csv(
            STABILITY_DIR / 'three_model_stability_summary.csv', index=False
        )
    figure_paths = create_main_figures(
        classification_results,
        faithfulness_metrics,
        deletion_curves,
        stability_metrics,
        CLASS_NAMES,
        FIGURE_DIR,
    )
    attribution_maps = pd.concat(
        [
            pd.read_csv(FAITHFULNESS_DIR / 'resnet18_attribution_maps.csv'),
            pd.read_csv(BASE_ARTIFACT_DIR / 'faithfulness' / 'uni_attribution_maps.csv'),
            pd.read_csv(FAITHFULNESS_DIR / 'dinov2_attribution_maps.csv'),
        ],
        ignore_index=True,
    )
    occlusion_scores = pd.concat(
        [
            pd.read_csv(FAITHFULNESS_DIR / 'resnet18_patch_occlusion_scores.csv'),
            pd.read_csv(BASE_ARTIFACT_DIR / 'faithfulness' / 'uni_patch_occlusion_scores.csv'),
            pd.read_csv(FAITHFULNESS_DIR / 'dinov2_patch_occlusion_scores.csv'),
        ],
        ignore_index=True,
    )
    heatmap_paths = create_three_model_heatmaps(
        cohort,
        attribution_maps,
        occlusion_scores,
        FIGURE_DIR / 'three_model_heatmaps',
        images_per_class=1,
    )
    for path in figure_paths + heatmap_paths:
        print(path)

## Interpretation guardrails

- The primary outcomes are occlusion Spearman correlation and top-minus-random true-class-margin AUC.
- A negative top-minus-random AUC indicates attribution-ranked deletion is more effective than random deletion.
- Raw-logit cross-model comparisons are secondary because logit scales differ.
- DINOv2 versus UNI compares general-domain and pathology-specific self-supervised pretraining with the same ViT scale, classifier design, attribution algorithm, images, and evaluation grid.
- ResNet versus either Transformer compares complete model–explanation pipelines, not architecture alone.
- Do not describe highlighted regions as biologically causal.